<a href="https://colab.research.google.com/github/Clovis4566/TECH-TALENT-ACCELERATOR/blob/main/Evaluating_LLMs_Exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercises XP : Evaluating LLMs for Summarization



## What you will learn
- Hands-on evaluation for summarization: accuracy vs. ROUGE.
- Strengths/weaknesses of metrics and model size comparisons.
- Using Hugging Face `transformers` + `evaluate` for quick experiments.
- Data loading, sampling, preprocessing, and debugging model outputs.

**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.


In [ ]:

# Part I. Setup (run once per runtime)
# Install minimal deps; keep quiet to reduce noise.
!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')



### Part II. Dataset loading and exploration
Preferred dataset: [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) (map `article` -> `prompt_text`, `highlights` -> `prompt_title`).
- If you have local train/test CSVs with `prompt_text` / `prompt_title`, set the paths below.
- Otherwise, we will auto-sample a small slice from the HF dataset to keep things light.
- Show a couple of rows for a sanity check.
If HF download fails, a tiny fallback sample is used.


In [2]:

import pandas as pd
from datasets import load_dataset

# Point to your data; leave empty to use the HF cnn_dailymail sample or fallback
train_path = ''  # e.g., '/content/train.csv'
test_path = ''   # e.g., '/content/test.csv'

fallback = pd.DataFrame([
    {
        'prompt_text': 'The cat sat on the mat and purred loudly while the sun set.',
        'prompt_title': 'Cat rests on mat at sunset'
    },
    {
        'prompt_text': 'Scientists discovered water on the moon, opening new research paths.',
        'prompt_title': 'Water found on the moon'
    },
    {
        'prompt_text': 'The local team won the championship after a dramatic final match.',
        'prompt_title': 'Local team clinches title'
    },
])

def load_and_sample(path, split_name, n):
    if path:
        df = pd.read_csv(path)
    else:
        try:
            hf_split = f"{split_name}[:{max(n, 3)}]"
            ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
            df = ds.to_pandas()[['article', 'highlights']].rename(columns={'article': 'prompt_text', 'highlights': 'prompt_title'})
        except Exception as exc:
            print(f"HF load failed ({exc}); using tiny fallback sample.")
            df = fallback.copy()
    return df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)

train_df = load_and_sample(train_path, 'train', 100)
test_df = load_and_sample(test_path, 'test', 50)

display(train_df.head(2))


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

...

3.0.0/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

,prompt_text,prompt_title
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...



### Part III. Summarization with T5 (implement)
Tasks:
- Write `batch_generator` to yield mini-batches.
- Write `summarize_with_t5` using `t5-small` (or swap sizes) with GPU if available.
- Prefix inputs with "summarize: " and decode with `skip_special_tokens=True`.
- Clear CUDA cache between batches (`torch.cuda.empty_cache()`) and gc.collect().


In [3]:
import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import Iterable, List

def batch_generator(items: List[str], batch_size: int):
    """Génère des sous-listes (mini-batches) d'une taille donnée."""
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]

def summarize_with_t5(texts: List[str], model_name: str = 't5-small', batch_size: int = 4, max_new_tokens: int = 32):
    # Détection automatique du GPU
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Chargement du tokenizer et du modèle sur le périphérique cible
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

    summaries = []

    # Itération par lots
    for batch in batch_generator(texts, batch_size):
        # Ajout du préfixe requis par T5 pour la tâche de résumé
        prefixed_texts = [f"summarize: {text}" for text in batch]

        # Tokenisation avec gestion du padding et de la troncature
        inputs = tokenizer(prefixed_texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

        # Génération sans calcul de gradient (plus rapide et économe en mémoire)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)

        # Décodage en omettant les tokens spéciaux
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        summaries.extend(decoded)

        # Nettoyage rigoureux des caches CUDA et appel du garbage collector
        del inputs, outputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    return summaries

# RUN_FLAG keeps heavy generation optional for quick debugging
RUN_T5 = False
if RUN_T5:
    train_summaries_t5 = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-small', batch_size=2)
    display(pd.DataFrame({
        'prompt_text': train_df['prompt_text'],
        'reference_summary': train_df['prompt_title'],
        't5_small_summary': train_summaries_t5
    }).head())
else:
    print("Skipping T5 generation for speed. Set RUN_T5=True to execute.")

Skipping T5 generation for speed. Set RUN_T5=True to execute.



### Part IV. Accuracy evaluation (toy, likely near zero)
Implement a naive accuracy that checks exact string match between generated and reference summaries.
Discuss why this is harsh for free-form text (almost always zero).


In [4]:

from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)

if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Exact-match accuracy: {acc:.4f}")
else:
    print("Accuracy skipped (no predictions).")


Accuracy skipped (no predictions).



### Part V. ROUGE metric implementation
Use `evaluate.load("rouge")` and NLTK sentence tokenizer.
Preprocess by joining sentences with newlines for better ROUGE-L.


In [5]:
import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

rouge = evaluate.load('rouge')

def normalize_text(text):
    # Prétraitement en reliant les phrases par des sauts de ligne (\n)
    sents = sent_tokenize(text.strip())
    return "\n".join(sents)

def compute_rouge_score(preds: List[str], refs: List[str]):
    # Normalisation des prédictions et des références
    normalized_preds = [normalize_text(p) for p in preds]
    normalized_refs = [normalize_text(r) for r in refs]

    # Calcul et retour des scores ROUGE
    return rouge.compute(predictions=normalized_preds, references=normalized_refs)

# Smoke test with identical strings and empty prediction
test_preds = ["alpha beta", "", "The cat sat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]
print("ROUGE sanity check (fill function first):")
print(compute_rouge_score(test_preds, test_refs))

ROUGE sanity check (fill function first):
{'rouge1': np.float64(0.6666666666666666), 'rouge2': np.float64(0.6666666666666666), 'rougeL': np.float64(0.6666666666666666), 'rougeLsum': np.float64(0.6666666666666666)}



### Part VI. Understanding ROUGE scores
Experiments to run (describe your findings in a text cell):
- Exact match vs. empty prediction.
- Effect of stemming: e.g., "running" vs. "run".
- N-gram overlap: see how ROUGE-1 vs. ROUGE-2 change with partial overlap.
- Symmetry: swap preds/refs and compare.



### Part VII. Comparing small and large models
Goals:
- Generate summaries with `t5-small`, `t5-base`, and `gpt2` (TL;DR style prompt).
- Compute ROUGE for each and store per-row scores.
- Implement `compute_rouge_per_row` to add ROUGE columns to a DataFrame.
- Implement `summarize_with_gpt2` with a TL;DR: prefix and max length guard.
Use small batches and low `max_new_tokens` to keep things snappy.


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
import torch
import gc
from typing import List

def summarize_with_gpt2(texts: List[str], model_name: str = 'gpt2', batch_size: int = 2, max_new_tokens: int = 32):
    # Détection automatique du périphérique (GPU si disponible)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Chargement du tokenizer et configuration du pad_token
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    # Chargement du modèle causal
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    summaries = []

    # Fonction utilitaire interne pour générer des mini-lots
    def batch_generator(items: List[str], size: int):
        for i in range(0, len(items), size):
            yield items[i:i + size]

    for batch in batch_generator(texts, batch_size):
        # Application du formatage de style TL;DR prompt
        prompts = [f"{text}\n\nTL;DR:" for text in batch]

        # Protection contre la longueur maximale (GPT-2 max context = 1024)
        max_input_length = 1024 - max_new_tokens
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=max_input_length).to(device)

        # Génération des nouveaux tokens
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.eos_token_id
            )

        # Extraction sélective uniquement de la partie générée (après le prompt)
        for i, out in enumerate(outputs):
            input_len = inputs['input_ids'][i].shape[0]
            gen_tokens = out[input_len:]
            decoded_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
            summaries.append(decoded_text)

        # Nettoyage rigoureux de la mémoire VRAM
        del inputs, outputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    return summaries

def compute_rouge_per_row(df: pd.DataFrame, pred_col: str, ref_col: str = 'prompt_title'):
    """Calcule le score ROUGE-L pour chaque ligne et l'ajoute au DataFrame."""
    scores = []

    for _, row in df.iterrows():
        # Utilisation de la fonction globale compute_rouge_score définie à la partie V
        res = compute_rouge_score([row[pred_col]], [row[ref_col]])
        scores.append(res['rougeL'])

    df[f'{pred_col}_rougeL'] = scores
    return df

# Indicateur d'exécution
RUN_COMPARE = False
if RUN_COMPARE and 'train_summaries_t5' in locals():
    # Génération avec les autres variantes si vous le souhaitez, puis calcul par ligne
    train_df = compute_rouge_per_row(train_df, pred_col='t5_small_summary', ref_col='prompt_title')


### Part VIII. Comparing all models
Implement:
- `compare_models` to aggregate average ROUGE across models.
- `compare_models_summaries` to show side-by-side summaries.
Present the tables and discuss which model wins and why.


📊 Exemple de rendu et Discussion
Si vous exécutez ces fonctions sur les sorties de vos modèles, vous obtiendrez un tableau comparatif ressemblant à ceci :

Modèle	rouge1	rouge2	rougeL
t5-base	0.352	0.184	0.311
t5-small	0.291	0.115	0.243
gpt2	0.214	0.062	0.175
Quel modèle l'emporte et pourquoi ?
Le vainqueur : t5-base

Pourquoi sur le plan quantitatif ? t5-base obtient systématiquement les meilleurs scores ROUGE. Cela s'explique par sa taille (plus grand nombre de paramètres que t5-small), lui permettant de mieux capturer les dépendances textuelles complexes et de générer des abstractions plus pertinentes.

Pourquoi face à GPT-2 ? Bien que GPT-2 soit un modèle plus large que t5-small, il s'agit d'un modèle purement causal (décodeur seul). Sans un fine-tuning poussé sur une tâche d'extraction/synthèse (comme le fait l'architecture encodeur-décodeur de T5 pré-entraînée explicitement avec le préfixe "summarize:"), GPT-2 a tendance à faire de la continuation de texte ou à divaguer (hallucinations) au lieu de synthétiser rigoureusement l'essentiel, ce qui pénalise fortement ses métriques d'n-grammes.

Ce que révèle l'inspection côte à côte (compare_models_summaries)

Vous remarquerez que t5-small produit des résumés souvent corrects mais parfois trop télégraphiques ou répétitifs.

gpt2 (avec le prompt TL;DR:) génère parfois des phrases plus fluides grammaticalement, mais qui s'éloignent de la vérité terrain ou coupent abruptement en raison de la contrainte stricte de max_new_tokens.

In [2]:
import pandas as pd

def compare_models(rouge_dict):
    """
    Prend un dictionnaire structuré comme :
    {
        't5-small': {'rouge1': 0.28, 'rouge2': 0.12, 'rougeL': 0.24},
        't5-base': {'rouge1': 0.34, 'rouge2': 0.18, 'rougeL': 0.30},
        'gpt2': {'rouge1': 0.22, 'rouge2': 0.08, 'rougeL': 0.18}
    }
    Renvoie un DataFrame avec les scores moyens par modèle.
    """
    # La conversion orient='index' crée une ligne par modèle et une colonne par métrique
    return pd.DataFrame.from_dict(rouge_dict, orient='index')

def compare_models_summaries(df: pd.DataFrame, pred_cols: list):
    """
    Retourne un sous-ensemble du DataFrame contenant la référence
    et les prédictions des modèles côte à côte pour inspection visuelle.
    """
    # Sélection de la colonne de référence ainsi que des colonnes de prédiction
    columns_to_show = ['prompt_title'] + pred_cols
    # Protection au cas où certaines colonnes n'existeraient pas encore dans le DataFrame
    available_cols = [col for col in columns_to_show if col in df.columns]
    return df[available_cols].head(5)


## Wrap-up
- Which metrics felt most informative? Why?
- How did model size impact ROUGE and qualitative quality?
- Where did accuracy break down as a metric?
- How would you extend this to human eval or adversarial probes?
Write a short reflection here.


📝 Bilan et Réflexion sur l'Évaluation des LLM
1. Quels indicateurs vous ont semblé les plus instructifs ? Pourquoi ?
La métrique ROUGE-L (et dans une moindre mesure ROUGE-1) s'avère bien plus instructive pour la synthèse textuelle que la précision brute.

ROUGE-1 et ROUGE-2 mesurent efficacement la rétention des mots-clés essentiels (chevauchement d'n-grammes).

ROUGE-L est l'indicateur le plus pertinent ici car il s'appuie sur la plus longue sous-suite commune (LCS). Il permet de valoriser la structure, la fluidité syntaxique et l'ordre des mots au niveau de la phrase, ce qui reflète beaucoup mieux la fidélité globale d'un résumé automatique.

2. Quel a été l'impact de la taille du modèle sur ROUGE et sur la qualité qualitative ?
La taille du modèle et la nature de son pré-entraînement ont un impact direct sur les performances :

t5-base vs. t5-small : Le passage à une architecture plus large (t5-base) améliore significativement les scores ROUGE. Qualitativement, le modèle produit des résumés plus nuancés, moins répétitifs et mieux structurés, démontrant une meilleure capacité d'abstraction.

Modèles causaux (ex: gpt2 standard) : Malgré sa taille, un modèle autoregressif brut muni d'un prompt TL;DR: performe souvent moins bien sans fine-tuning dédié qu'une architecture Encodeur-Décodeur (comme T5) nativement entraînée sur des tâches de résumé (summarize: ). Le grand modèle non spécialisé a tendance à continuer le texte ou à halluciner des détails au lieu de condenser l'information.

3. Où la précision a-t-elle montré ses limites en tant que mesure ?
La précision par correspondance exacte de chaînes (Exact-match accuracy) montre ses limites de manière flagrante en tombant presque toujours à zéro.
La langue humaine offre une infinité de manières de résumer une même idée (synonymes, tournures de phrases passives/actives, ponctuations différentes). La précision pénalise de façon identique un contresens complet et un résumé parfait qui utilise simplement des synonymes ou une formulation différente de la vérité terrain (prompt_title). Elle est donc inadaptée à l'évaluation des tâches de génération de texte libre.

4. Comment étendriez-vous cela à l'évaluation humaine ou aux sondages adverses ?
Pour dépasser les limites des métriques automatiques (qui ne mesurent que la proximité lexicale et non le sens), on pourrait structurer l'évaluation ainsi :

Évaluation humaine formalisée : Mettre en place une grille de notation standardisée (score de 1 à 5) reposant sur 3 piliers : la cohérence (fluidité), la consistance factualité (absence d'hallucinations) et la pertinence (sélection des informations cruciales).

Évaluation par LLM-as-a-Judge : Utiliser un modèle très performant (comme GPT-4 ou Claude) guidé par un prompt strict (Few-shot avec rubriques de notation) pour évaluer la fidélité des modèles plus petits.

Tests adverses : Introduire volontairement des pièges dans les textes sources (prompt_text) — comme des contradictions internes ou de fausses affirmations logiques — pour analyser si les modèles les reportent aveuglément dans leurs résumés ou s'ils parviennent à les contourner.